# OCR ve Doküman Zekâsı Motor Karşılaştırması — Gerçek Derlem & StudyApp Modelleri

Bu notebook; **StudyApp** projesinde kıyaslanan ve **KACHOW** üretim hattında yer alan doküman işleme motorlarını (`OpenDataLoader-PDF`, `Tesseract (300 DPI + Layout)`, `PaddlePaddle/PaddleOCR`, `ATH-MaaS/OvisOCR2`, `zai-org/GLM-OCR`, `baidu/Unlimited-OCR`, `deepseek-ocr:latest`), **23 gerçek taranmış resmî yazışma belgesi (52 sayfa)** üzerinde kıyaslar.

### İki Boyutlu Kapsamlı Değerlendirme:
1. **StudyApp Ticari Değiş-Tokuş Matrisi (Latency vs. Accuracy vs. Memory Footprint):** Sayfa başına çıkarım süresi (gecikme), tepe bellek kullanımı (RAM/VRAM) ve ağırlıklı başarı skoru üzerinden baloncuk dağılım grafiği (**Scatter Bubble Plot**).
2. **KACHOW Alan Düzeyinde Ayrışma (Header vs. Signature Recovery):** Başlık bölgesi (Sayı/Tarih/Konu/Muhatap/Gönderen) ve ıslak imzanın basılı ismi yok ettiği imza bölgesi (İmza sahibi/unvanı) için çift çubuk grafik (**Dual Bar Chart**).

Skorlama mantığı `scripts/evaluate_ocr_real.py`'de yaşar ve üretim kodunu doğrudan import eder.

**Çalıştırmadan önce**: Colab menüsünden *Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU* seçili olmalı. Ayrıca sol paneldeki 🔑 (Secrets) sekmesine `GITHUB_TOKEN` adıyla bir token eklenmelidir.

## 1. GPU Doğrulama

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
print()
print("Yukarıda 'A100' görmüyorsanız: Çalışma zamanı > Çalışma zamanı türünü değiştir > A100 GPU, sonra bu hücreyi tekrar çalıştırın.")

## 2. Repo Klonu

`GITHUB_TOKEN` Colab Secrets'tan okunur, hiçbir hücreye açık yazılmaz. İmza alanı skorlaması ve StudyApp modelleri `fix/ocr-benchmark-signature-scoring` dalında bulunmaktadır.

In [ ]:
from google.colab import userdata
import os

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
REPO = "chyp3r/KACHOW-Teknofest-2026"
BRANCH = "fix/ocr-benchmark-signature-scoring"  # main'e merge edildiyse 'main' yapabilirsiniz

clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"
!git clone --branch {BRANCH} --depth 1 {clone_url} repo 2>&1 | sed "s/{GITHUB_TOKEN}/***/g"
%cd repo
!git log -1 --oneline -- scripts/evaluate_ocr_real.py

## 3. Sistem Paketleri

Backend Docker ortamıyla birebir aynı: Java (OpenDataLoader PDF için), Tesseract OCR ve Türkçe dil paketi.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y default-jre-headless tesseract-ocr tesseract-ocr-tur
!java -version
!tesseract --version | head -1
!tesseract --list-langs | grep -q tur && echo 'tur dil paketi OK' || echo 'HATA: tur dil paketi eksik'

## 4. Python Bağımlılıkları

Backend gereksinimleri, event loop uyumluluğu için `nest_asyncio` ve StudyApp modelleri için `paddlepaddle`, `paddleocr` kurulur.

In [ ]:
!pip install -q -r backend/requirements.txt nest_asyncio paddlepaddle paddleocr matplotlib
import sys
sys.path.insert(0, "backend")

import nest_asyncio
nest_asyncio.apply()

from app.infrastructure.extractors import FallbackDocumentExtractor  # noqa: F401
print("Backend paketleri ve bağımlılıklar OK")

## 5. Ollama & Model Hazırlığı

Ollama arka planda başlatılır ve VLM modelleri (`glm-ocr`, `deepseek-ocr`, `unlimited-ocr`) çekilir.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request, urllib.error

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for attempt in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=1)
        print(f"Ollama hazır ({attempt + 1}. denemede).")
        break
    except (urllib.error.URLError, ConnectionError):
        time.sleep(1)
else:
    raise RuntimeError("Ollama 30 saniyede ayağa kalkmadı -- `!ollama serve` çıktısını kontrol edin.")

!ollama --version

In [ ]:
import subprocess
for model in ["glm-ocr:latest", "deepseek-ocr:latest", "frob/unlimited-ocr:q8_0"]:
    print(f"--- çekiliyor: {model} ---")
    subprocess.run(["ollama", "pull", model], check=True)
!ollama list

## 6. Duman Testi (Kalite Kapısı)

Tek zorlu belge (`CY-050` — imza basılı ismi bozan belge) ve `glm-ocr` ile duman testi koşulur.

In [ ]:
import asyncio, importlib.util, logging
logging.basicConfig(level=logging.INFO)

spec = importlib.util.spec_from_file_location("ev", "scripts/evaluate_ocr_real.py")
ev = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ev)

gt = ev._load_ground_truth()
docs = ev._load_documents(gt)
doc = [d for d in docs if d[0].startswith("CY-050")][0]
print("beklenen alanlar:", sorted(doc[2].keys()), f"-- {len(doc[2])} alan (imza dahil)")
assert len(doc[2]) >= 5 and any(k.startswith("imza") for k in doc[2]), "İmza alanları skorlamada yok -- repo dalının fix/ocr-benchmark-signature-scoring olduğunu doğrulayın!"

name, engine = ev._parse_engine_spec("ollama:glm-ocr:latest")
smoke = asyncio.run(ev._run_engine(name, engine, [doc]))
sig = smoke["totals"]["chain"]["signature"]
print(f"\nzincir imza sonucu: {sig['found']}/{sig['expected']} bulundu")
print("Log'da 'full-page vision transcription' görüldüyse kapı geçti -- 7. hücreye geçebilirsiniz.")

## 7. Tam Koşum (StudyApp Modelleri Dahil)

Aşağıdaki modeller sırayla değerlendirilir ve sonuçlar `ocr_real_results.json`'a birikimli yazılır:
- `opendataloader`: Java tabanlı OpenDataLoader-PDF (~1-2 dk)
- `tesseract`: Tesseract (300 DPI + Layout) (~2-3 dk)
- `paddleocr`: PaddlePaddle/PaddleOCR (~1-2 dk)
- `ollama:glm-ocr:latest`: zai-org/GLM-OCR (~30-40 dk)
- `ollama:frob/unlimited-ocr:q8_0`: Baidu Unlimited-OCR (~30-40 dk)
- `ollama:deepseek-ocr:latest`: DeepSeek-OCR (~35-45 dk)
- *(Opsiyonel)* `sidecar:ovisocr2`: ATH-MaaS/OvisOCR2

> **NOT:** Tüm modellerin tamamlanması toplam ~2-2.5 saat sürer. Her motor tamamlandıkça sonuç diske kaydedilir.

In [ ]:
import sys
sys.argv = [
    "evaluate_ocr_real.py",
    "--engine", "opendataloader",
    "--engine", "tesseract",
    "--engine", "paddleocr",
    "--engine", "ollama:glm-ocr:latest",
    "--engine", "ollama:frob/unlimited-ocr:q8_0",
    "--engine", "ollama:deepseek-ocr:latest",
]
ev.main()

## 8. Ara/Son Durum Tablosu

In [ ]:
results = ev._load_results_file(ev.DEFAULT_RESULTS_FILE)
print(f"Tamamlanan motorlar: {list(results.keys())}\n")
ev._print_summary(results)

## 9. Görselleştirme 1: StudyApp Latency vs. Accuracy Baloncuk Grafiği

**X Ekseni:** Zincir (üretim yolu) Sayfa Başı Gecikme (saniye) — imza/başlık kurtarma
için yapılan ek vision çağrılarını da içerir; motorun ham tek-çağrı hızı ayrı olarak
etiketlerde `ham Xs` şeklinde gösterilir. [Düşük olması daha iyi]
**Y Ekseni:** Ağırlıklı Doküman Zekâsı Doğruluğu (%) [Yüksek olması daha iyi]
**Baloncuk Boyutu:** Tahmini Bellek Kullanımı (MB) — **ölçülmedi**, model adına göre
elle yazılmış kaba bir referans tablosundan (`ESTIMATED_ENGINE_MEMORY_MB`) geliyor.
**Yeşil Alan (Sweet Spot Quadrant):** Zincir gecikmesi < 7.0s & Doğruluk > 90% hedef bölgesi (GLM-OCR şampiyon).


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Model etiketleri ve renk eşleştirmeleri
ENGINE_PRETTY_NAMES = {
    "opendataloader": "OpenDataLoader-PDF",
    "tesseract": "Tesseract (300 DPI + Layout)",
    "paddleocr": "PaddlePaddle/PaddleOCR",
    "ollama:glm-ocr:latest": "zai-org/GLM-OCR",
    "ollama:frob/unlimited-ocr:q8_0": "baidu/Unlimited-OCR",
    "ollama:deepseek-ocr:latest": "deepseek-ocr",
    "ovisocr2": "ATH-MaaS/OvisOCR2",
}

ENGINE_COLORS = {
    "OpenDataLoader-PDF": "#10b981",
    "OpenCV + PyZBar": "#3b82f6",
    "Tesseract (300 DPI + Layout)": "#6366f1",
    "PaddlePaddle/PaddleOCR": "#ec4899",
    "ATH-MaaS/OvisOCR2": "#8b5cf6",
    "zai-org/GLM-OCR": "#f97316",
    "baidu/Unlimited-OCR": "#eab308",
    "deepseek-ocr": "#06b6d4",
}

# Dinamik Eksen Aralıkları
latencies = [res.get("chain_seconds_per_page", 0.5) for res in results.values()]
accuracies = [res.get("weighted_accuracy_pct", 75.0) for res in results.values()]
max_lat = max(max(latencies), 4.5) * 1.15 if latencies else 5.0
min_acc = max(0.0, min(min(accuracies), 40.0) - 8.0) if accuracies else 25.0

# Sweet Spot Eşikleri: GLM-OCR (~6.2s, %93.9) gibi yüksek başarılı modelleri kapsayacak şekilde
SWEET_SPOT_MAX_LAT = 7.5
SWEET_SPOT_MIN_ACC = 90.0

fig, ax = plt.subplots(figsize=(11, 6.5), facecolor="#ffffff")
ax.set_facecolor("#ffffff")

# 1. Hedef Bölge (Sweet Spot Quadrant): Latency < 7.5s, Accuracy > 90%
ax.axhspan(SWEET_SPOT_MIN_ACC, 105, xmin=0, xmax=(SWEET_SPOT_MAX_LAT / max_lat), color="#eef8f1", alpha=0.9, zorder=0)
ax.axvline(SWEET_SPOT_MAX_LAT, color="#a3b899", linestyle="--", linewidth=1.2, alpha=0.8, zorder=1)
ax.axhline(SWEET_SPOT_MIN_ACC, color="#a3b899", linestyle="--", linewidth=1.2, alpha=0.8, zorder=1)
ax.text(0.2, 91.5, f"Sweet Spot (Hedef Bölge)\nLatency < {SWEET_SPOT_MAX_LAT:.1f}s | Doğruluk > {SWEET_SPOT_MIN_ACC:.0f}%", fontsize=9, color="#2e6930", weight="bold", zorder=2)

# 2. Noktaları Çiz
for raw_name, res in results.items():
    pretty_name = ENGINE_PRETTY_NAMES.get(raw_name, raw_name)
    ch = res["totals"]["chain"]
    h_found, h_exp = ch["header"]["found"], ch["header"]["expected"]
    s_found, s_exp = ch["signature"]["found"], ch["signature"]["expected"]
    
    # Ağırlıklı doğruluk (%60 Başlık, %40 İmza)
    acc = res.get("weighted_accuracy_pct")
    if acc is None:
        h_pct = (100 * h_found / h_exp) if h_exp else 0
        s_pct = (100 * s_found / s_exp) if s_exp else 0
        acc = 0.6 * h_pct + 0.4 * s_pct
        
    # chain_seconds_per_page = üretimde gerçekten ödenen maliyet (imza/başlık
    # kurtarma yeniden denemeleri dahil); raw_seconds_per_page = motorun tek
    # çağrıdaki ham hızı. Bir motorun kurtarma zinciri daha SIK başarıyla
    # tetiklenmesi (daha çok deniyor ve kazanıyor olması) onu bu grafikte
    # "yavaş" gösterebilir -- ham hız ayrıca etikette gösteriliyor.
    lat = res.get("chain_seconds_per_page", 0.5)
    raw_lat = res.get("raw_seconds_per_page", lat)
    mem = res.get("estimated_memory_mb", 1000.0)
    color = ENGINE_COLORS.get(pretty_name, "#2a78d6")
    
    # Baloncuk boyutu bellek ile orantılı
    bubble_size = (mem / 3500.0) * 850 + 100
    
    ax.scatter(lat, acc, s=bubble_size, color=color, alpha=0.8, edgecolors="#000000", linewidth=1.2, zorder=3)
    
    # Çakışmaları önlemek için özel etiket ofsetleri
    offset_x, offset_y = 10, 8
    if "deepseek" in raw_name or "deepseek" in pretty_name:
        offset_x, offset_y = 12, -18
    elif "GLM" in pretty_name or "glm" in raw_name:
        offset_x, offset_y = -35, 16
    elif "Ovis" in pretty_name:
        offset_x, offset_y = -45, 15
    elif "Paddle" in pretty_name:
        offset_x, offset_y = -50, 14
    elif "OpenDataLoader" in pretty_name:
        offset_x, offset_y = 10, -22
    elif "Tesseract" in pretty_name:
        offset_x, offset_y = 10, 10
    elif "unlimited" in raw_name or "Unlimited" in pretty_name:
        offset_x, offset_y = -65, 12
        
    ax.annotate(
        f"{pretty_name}\n(~{mem:.0f} MB tahmini, ham {raw_lat:.1f}s)",
        (lat, acc),
        xytext=(offset_x, offset_y),
        textcoords="offset points",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#cccccc", alpha=0.9),
        fontsize=8.5,
        weight="bold",
        zorder=4,
    )

# Eksen & Izgara Ayarları
ax.set_xlim(0.0, max_lat)
ax.set_ylim(min_acc, 105.0)
ax.set_xlabel("Chain Latency (Seconds per Page, incl. recovery retries) [Lower is Better]", fontsize=11, fontweight="medium", labelpad=10)
ax.set_ylabel("Weighted Document Intelligence Accuracy (%) [Higher is Better]", fontsize=11, fontweight="medium", labelpad=10)
ax.set_title("Chain Latency vs. Document Intelligence Accuracy\n(Bubble size: ESTIMATED memory footprint in MB, not measured)", fontsize=13, fontweight="bold", pad=15)

ax.grid(True, linestyle=":", alpha=0.6, color="#cbd5e1", zorder=1)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig("ocr_bubble_benchmark.svg", format="svg", bbox_inches="tight")
plt.savefig("ocr_bubble_benchmark.png", format="png", dpi=200, bbox_inches="tight")
plt.show()
print("\nKaydedildi: ocr_bubble_benchmark.svg, ocr_bubble_benchmark.png")

## 10. Görselleştirme 2: KACHOW Başlık ve İmza Kurtarma Çift Çubuk Grafiği

In [ ]:
COLOR_HEADER = "#2a78d6"
COLOR_SIGNATURE = "#eb6834"
TEXT_PRIMARY = "#0b0b0b"
TEXT_SECONDARY = "#52514e"

engines = list(results.keys())
header_pct, signature_pct = [], []
for name in engines:
    ch = results[name]["totals"]["chain"]
    header_pct.append(100 * ch["header"]["found"] / ch["header"]["expected"])
    signature_pct.append(100 * ch["signature"]["found"] / ch["signature"]["expected"])

x = np.arange(len(engines))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5.5), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

bars1 = ax.bar(x - width / 2, header_pct, width, label="Başlık alanları (Sayı/Tarih/Konu/Muhatap/Gönderen)",
               color=COLOR_HEADER, edgecolor="none")
bars2 = ax.bar(x + width / 2, signature_pct, width, label="İmza alanları (İmza sahibi/unvanı)",
               color=COLOR_SIGNATURE, edgecolor="none")

for bars in (bars1, bars2):
    for b in bars:
        ax.annotate(f"{b.get_height():.0f}%", (b.get_x() + b.get_width() / 2, b.get_height()),
                    ha="center", va="bottom", fontsize=9, color=TEXT_PRIMARY)

ax.set_ylim(0, 108)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_ylabel("Alan bulma oranı", color=TEXT_SECONDARY)
ax.set_xticks(x)
ax.set_xticklabels(engines, rotation=15, ha="right", color=TEXT_PRIMARY)
ax.set_title("Zincir bazında OCR alan kurtarma — gerçek 23 belge / 52 sayfa",
             color=TEXT_PRIMARY, fontsize=13, pad=14)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color("#d8d7d0")
ax.tick_params(colors=TEXT_SECONDARY, length=0)
ax.grid(axis="y", color="#e8e7e0", linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False, fontsize=9, labelcolor=TEXT_SECONDARY)

plt.tight_layout()
plt.savefig("ocr_benchmark.svg", format="svg", bbox_inches="tight")
plt.savefig("ocr_benchmark.png", format="png", dpi=200, bbox_inches="tight")
plt.show()
print("\nKaydedildi: ocr_benchmark.svg, ocr_benchmark.png")

## 11. İndir

Ham JSON'u ve her iki grafiği (StudyApp baloncuk grafiği + KACHOW çift çubuk grafiği) indirin.

In [ ]:
from google.colab import files
files.download(ev.DEFAULT_RESULTS_FILE)
files.download("ocr_bubble_benchmark.svg")
files.download("ocr_bubble_benchmark.png")
files.download("ocr_benchmark.svg")
files.download("ocr_benchmark.png")